# Lab 17: Projet Final - Pipeline DS-STAR Complet

**Navigation** : [Lab 16 <<](Lab16-Data-Science-Agent.ipynb) | [Index](../../README.md)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Intégrer tous les composants DS-STAR : FileAnalyzer, Planner-Coder-Verifier
2. Appliquer à un problème réel de data science
3. Générer un rapport d'analyse automatisé complet
4. Déployer un agent de data science fonctionnel

### Prérequis
- Tous les labs précédents (Lab 8-16) complétés
- Projet GCP configuré (optionnel)
- Maîtrise des patterns d'agents

### Durée estimée : 60-90 minutes

---

## Architecture Complète

```
Fichiers CSV/JSON --> FileAnalyzer --> Planner --> Coder --> Executor --> Verifier --> Reporter
```

## 1. Configuration

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import re
import pandas as pd
import numpy as np
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, asdict
from enum import Enum
from pathlib import Path

from config import get_settings
from utils import LLMClient

Chargement des parametres de configuration.

In [ ]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

## 2. Data Classes

In [ ]:
@dataclass
class Plan:
    steps: List[str]
    reasoning: str

@dataclass
class ExecutionResult:
    success: bool
    output: str
    error: Optional[str] = None
    code: Optional[str] = None

@dataclass
class FileMetadata:
    filename: str
    format: str
    size_bytes: int
    num_rows: int = None
    num_columns: int = None
    columns: list = None

class VerificationStatus(Enum):
    SUCCESS = 'success'
    NEEDS_REFINEMENT = 'needs_refinement'
    FAILED = 'failed'

## 3. Modules DS-STAR

In [ ]:
class FileAnalyzer:
    SUPPORTED = {'.csv': 'csv', '.json': 'json'}

    def __init__(self, llm=None):
        self.llm = llm or LLMClient()

    def analyze_csv(self, path: str) -> FileMetadata:
        df = pd.read_csv(path)
        cols = [{'name': c, 'dtype': str(df[c].dtype)} for c in df.columns]
        return FileMetadata(
            filename=Path(path).name, format='csv', size_bytes=os.path.getsize(path),
            num_rows=len(df), num_columns=len(df.columns), columns=cols
        )

    def generate_context(self, meta: FileMetadata) -> str:
        lines = [f'Fichier: {meta.filename}', f'Lignes: {meta.num_rows}', f'Colonnes: {meta.num_columns}']
        if meta.columns:
            lines.append('Colonnes: ' + ', '.join([c['name'] for c in meta.columns[:5]]))
        return '\\n'.join(lines)

Composant FileAnalyzer et pipeline d'orchestration.

In [ ]:
class Planner:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def create_plan(self, question: str, context: str) -> Plan:
        prompt = f'Planifie: {question}\\nCONTEXTE: {context[:600]}\\nDonne 2-3 etapes. Format: REASONING: [...] STEPS: 1. [...] 2. [...]'
        response = self.llm.generate(prompt, temperature=0.3)
        steps, reasoning = [], ''
        in_steps = False
        for line in response.split('\\n'):
            if line.startswith('REASONING:'):
                reasoning = line.replace('REASONING:', '').strip()
            elif line.startswith('STEPS:'):
                in_steps = True
            elif in_steps and re.match(r'^\\d+\\.', line.strip()):
                steps.append(re.sub(r'^\\d+\\.\\s*', '', line.strip()))
        return Plan(steps=steps, reasoning=reasoning)

Composant Planner : decomposition du probleme en etapes.

In [ ]:
class Coder:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def generate_code(self, plan: Plan, context: str) -> str:
        steps_text = '\\n'.join(f'{i+1}. {s}' for i, s in enumerate(plan.steps))
        prompt = f'Code Python pour: {steps_text}\\nDataFrame: df. Utilise print(). ```python [code] ```'
        response = self.llm.generate(prompt, temperature=0.2)
        match = re.search(r'```python\\s*(.*?)\\s*```', response, re.DOTALL)
        return match.group(1).strip() if match else ''

Composant Coder : generation de code d'analyse.

In [ ]:
class Executor:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.namespace = {'df': df, 'pd': pd, 'np': np, 'print': print}

    def execute(self, code: str) -> ExecutionResult:
        from io import StringIO
        import sys
        old_stdout = sys.stdout
        sys.stdout = StringIO()
        try:
            exec(code, self.namespace)
            return ExecutionResult(success=True, output=sys.stdout.getvalue(), code=code)
        except Exception as e:
            return ExecutionResult(success=False, output=sys.stdout.getvalue(), error=str(e), code=code)
        finally:
            sys.stdout = old_stdout

Composant Executor : execution sandbox du code genere.

In [ ]:
class Verifier:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def verify(self, question: str, result: ExecutionResult) -> Tuple[VerificationStatus, str]:
        if not result.success:
            return VerificationStatus.FAILED, f'Erreur: {result.error}'
        prompt = f'Verifie: {question}\\nResultat: {result.output[:400]}\\nReponds SUCCESS ou NEEDS_REFINEMENT.'
        response = self.llm.generate(prompt, temperature=0.1).upper()
        if 'SUCCESS' in response:
            return VerificationStatus.SUCCESS, 'OK'
        return VerificationStatus.NEEDS_REFINEMENT, 'A ameliorer'

Composant Verifier : validation des resultats par le LLM.

In [ ]:
class ReportGenerator:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def generate_report(self, question: str, results: List[Dict], metadata: FileMetadata) -> str:
        prompt = f'Genere un rapport Markdown pour: {question}\\nFichier: {metadata.filename} ({metadata.num_rows} lignes)\\nResultats: {len(results)} analyses'
        return self.llm.generate(prompt, temperature=0.3)

## 4. Pipeline Complet

In [ ]:
class DSStarPipeline:
    def __init__(self, max_iterations: int = 2):
        self.llm = LLMClient()
        self.file_analyzer = FileAnalyzer(self.llm)
        self.planner = Planner(self.llm)
        self.coder = Coder(self.llm)
        self.verifier = Verifier(self.llm)
        self.reporter = ReportGenerator(self.llm)
        self.max_iterations = max_iterations

    def analyze(self, file_path: str, question: str) -> Dict:
        print('='*50 + '\\nDS-STAR PIPELINE\\n' + '='*50)

        meta = self.file_analyzer.analyze_csv(file_path)
        context = self.file_analyzer.generate_context(meta)
        print(f'[FILE] {meta.filename}: {meta.num_rows} lignes')

        df = pd.read_csv(file_path)
        executor = Executor(df)
        results = []

        for i in range(self.max_iterations):
            print(f'\\n=== ITERATION {i+1} ===')
            plan = self.planner.create_plan(question, context)
            print(f'[PLANNER] {len(plan.steps)} etapes')
            code = self.coder.generate_code(plan, context)
            result = executor.execute(code)
            print(f'[EXECUTOR] {"OK" if result.success else "ERROR"}')
            if result.success:
                status, msg = self.verifier.verify(question, result)
                print(f'[VERIFIER] {status.value}')
                results.append({'output': result.output, 'status': status.value})
                if status == VerificationStatus.SUCCESS:
                    break
                context += f'\\nResultat: {result.output[:200]}'

        report = self.reporter.generate_report(question, results, meta)
        return {'success': len(results) > 0, 'results': results, 'report': report}

## 5. Dataset de Test

In [ ]:
import tempfile
test_dir = tempfile.mkdtemp()

np.random.seed(42)
df = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=200, freq='D'),
    'product': np.random.choice(['A', 'B', 'C', 'D'], 200),
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 200),
    'revenue': np.random.uniform(500, 5000, 200).round(2),
    'units': np.random.randint(5, 100, 200)
})

csv_path = os.path.join(test_dir, 'sales.csv')
df.to_csv(csv_path, index=False)
print(f'Dataset: {csv_path} ({len(df)} lignes)')

## 6. Execution du Pipeline

In [ ]:
pipeline = DSStarPipeline(max_iterations=1)
result = pipeline.analyze(csv_path, 'Analyse les ventes par region')

## 7. Resultats

In [ ]:
print(chr(10) + "="*50 + chr(10) + "RAPPORT" + chr(10) + "="*50)
print(result["report"][:800])

Nettoyage des fichiers temporaires generes pendant le pipeline.

In [ ]:
import shutil
shutil.rmtree(test_dir)
print('Done')

## 8. Conclusion
### Competences acquises
- Architecture d'agents pour data science
- Boucles iteratives avec raffinement
- Abstraction multi-provider
- Rapports automatises
### Extensions possibles
- Support de plus de formats
- Parallelisation
- Interface web
- Deploiement cloud
Felicitation! Vous avez complete la serie AgenticDataScience.

## Exercice Final : Pipeline DS-STAR sur Donnees Reelles

Cet exercice final vous guide pour appliquer le pipeline complet a un probleme reel.

### Objectifs
1. Trouver un dataset public (Kaggle, UCI, data.gouv)
2. Configurer le pipeline DS-STAR
3. Poser 3 questions d'analyse d'augmenter complexite
4. Generer un rapport final

### Instructions



In [ ]:
# TODO: Chargez un dataset reel
# Exemples: 
# - Kaggle: titanic, housing prices, credit card fraud
# - UCI: iris, wine, adult census
# - data.gouv: donnees ouvertes francaises

import pandas as pd
# mon_df = pd.read_csv('mon_dataset.csv')
# print(f"Dataset: {mon_df.shape}")

# TODO: Sauvegardez en CSV pour le pipeline
# mon_df.to_csv('data_temp.csv', index=False)

# TODO: Definissez 3 questions d'analyse
questions = [
    "Question simple - agregation de base",
    "Question intermediaire - comparaison de groupes",
    "Question complexe - analyse temporelle ou correlation"
]

# TODO: Executez le pipeline pour chaque question
# pipeline = DSStarPipeline(max_iterations=2)
# for q in questions:
#     result = pipeline.analyze('data_temp.csv', q)
#     print(f"\\n{q}: {result['success']}")

# TODO: Generez un rapport de synthese
#rapport_final = """
## Analyse du Dataset [Nom]
# 
# ### Questions posees
# 1. ...
# 2. ...
# 3. ...
#
# ### Resultats cles
# - ...
#
# ### Limitations
# - ...
#"""

### Extensions (bonus)
- Ajoutez des visualisations avec matplotlib/seaborn
- Exportez le rapport en Markdown ou HTML
- Integrez une validation croisee des resultats
- Connectez a une API externe pour enrichir les donnees
### Checklist de completion
- [ ] Dataset charge et inspecte
- [ ] 3 questions posees au pipeline
- [ ] Resultats analyses
- [ ] Rapport final genere
- [ ] (Bonus) Visualisations ajoutees
